## ▶ Cách chạy notebook này (khoảng 40 phút, không cần sửa code)
1. Đặt tên notebook (ô tên ở trên cùng): **`RMIT_SGS_GenData-b1-mmbert-small_C1`**
2. Cột phải → **Add Input** → tab *Competitions* → chọn **RMIT HACKATHON 2026**
3. **Add Input** → tab *Models* → tìm **`aisuko/mmbert-small`** → chọn biến thể *Transformers · default* → Add
4. Cột phải → **Session options** → *Accelerator* = **GPU T4 x2**
5. Bấm **Save Version** (góc trên phải) → chọn **Save & Run All (Commit)** → Save
6. Chờ khoảng 40 phút → mở bản version vừa chạy → ô cuối phải in `rc=0` cho cả 2 job
7. **Share** → thêm **`longquanton`** → nhắn lead. **Không nộp bài** từ tài khoản này.


# RMIT_SGS_GenData-b1-mmbert-small_C1

RMIT Hackathon 2026 · Challenge 1 — multilingual prompt-attack classification (BENIGN / JAILBREAK / PROMPT_INJECTION), metric: macro F1.

Benchmark: one model per GPU, in parallel.

- `mmbert-small`: **jhu-clsp/mmBERT-small** · folds [0, 1, 2] · lr 5e-05 · epochs 3 · max_len 512
- `mmbert-small`: **jhu-clsp/mmBERT-small** · folds [3, 4] · lr 5e-05 · epochs 3 · max_len 512

## Source code
The `rmitguard` package from our repository, written to disk so the run is reproducible.

In [ ]:
import os
os.makedirs('src/rmitguard', exist_ok=True)
os.makedirs('src/rmitguard/models', exist_ok=True)

In [ ]:
%%writefile src/rmitguard/__init__.py
"""Team GenData — RMIT Hackathon 2026, Challenge 1 (multilingual prompt-attack classification)."""

__version__ = "0.1.0"

In [ ]:
%%writefile src/rmitguard/__main__.py
import sys

from .cli import main

sys.exit(main())

In [ ]:
%%writefile src/rmitguard/charts.py
"""Charts for one run (training health + error inspection) and for comparing runs.

    plot_run(run_dir)                  -> run_dir/charts/*.png + run_dir/top_errors.csv
    plot_compare(run_dirs, out_dir)    -> out_dir/*.png + out_dir/summary.csv

Colours: the validated categorical palette of the dataviz reference (fixed order, never cycled).
Classes always use slots 1-3 (BENIGN blue, JAILBREAK orange, PROMPT_INJECTION aqua), which are
colour-blind safe as a set. More than 8 models are never coloured individually: small multiples instead.
"""

from __future__ import annotations

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .constants import LABELS
from .metrics import macro_f1

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
CLASS_COLOR = dict(zip(LABELS, SERIES[:3]))
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e0", "#fcfcfb"
BLUES = matplotlib.colors.LinearSegmentedColormap.from_list(
    "blues", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281"]
)
LENGTH_BUCKETS = [
    (0, 100, "<100"),
    (100, 300, "100–299"),
    (300, 800, "300–799"),
    (800, 2000, "800–1999"),
    (2000, 10**9, "2000+"),
]  # characters

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.labelsize": 9,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "legend.fontsize": 8,
        "font.size": 9,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)


def _save(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def _bars(ax, labels, values, color=SERIES[0], fmt="{:.3f}", xlim=None):
    """Dot plot (not bars): honest even when the axis does not start at zero. Values in ink."""
    y = np.arange(len(labels))
    ax.hlines(y, xlim[0] if xlim else 0, values, color=GRID, lw=1, zorder=1)
    ax.scatter(values, y, s=64, color=color, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    for yi, v in zip(y, values):
        if np.isfinite(v):
            ax.text(v, yi, "   " + fmt.format(v), va="center", fontsize=8, color=INK)
    if xlim:
        ax.set_xlim(*xlim)


def _f1_xlim(values) -> tuple[float, float]:
    v = np.asarray([x for x in values if np.isfinite(x)])
    lo = max(0.0, (v.min() if len(v) else 0.5) - 0.05)
    return (np.floor(lo * 20) / 20, 1.0 + (1.0 - lo) * 0.12)


# ------------------------------------------------------------------------------------------- one run
def load_predictions(run_dir: Path) -> dict[str, pd.DataFrame]:
    out = {}
    for name in ("holdout", "oof"):
        p = run_dir / f"{name}_predictions.csv"
        if p.exists():
            out[name] = pd.read_csv(p)
    return out


def plot_training_health(history: pd.DataFrame, title: str, path: Path) -> None:
    """Loss (train vs probe vs val), validation F1 per class, gradient norm, learning rate — per epoch."""
    h = history.copy()
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
    (a_loss, a_f1), (a_grad, a_lr) = axes
    folds = sorted(h["fold"].dropna().unique()) if "fold" in h else [0]

    def by_epoch(col):
        d = h.dropna(subset=[col]) if col in h else pd.DataFrame(columns=["epoch", col])
        return d.groupby("epoch")[col].agg(["mean", "min", "max"])

    curves = [
        ("loss", "train loss (logged while training)", SERIES[0]),
        ("eval_train_loss", "train loss (fixed train sample, eval mode)", SERIES[1]),
        ("eval_val_loss", "validation loss", SERIES[2]),
    ]
    for col, label, color in curves:
        m = by_epoch(col)
        if len(m):
            a_loss.plot(
                m.index,
                m["mean"],
                color=color,
                label=label,
                marker="o" if col != "loss" else None,
                ms=4,
                lw=2 if col != "loss" else 1.2,
            )
            if len(folds) > 1:
                a_loss.fill_between(m.index, m["min"], m["max"], color=color, alpha=0.12, lw=0)
    a_loss.set(
        title="Loss — gap between orange and aqua = overfitting", xlabel="epoch", ylabel="cross-entropy"
    )
    a_loss.legend(loc="upper right")

    m = by_epoch("eval_val_macro_f1")
    if len(m):
        a_f1.plot(m.index, m["mean"], color=INK, lw=2.4, label="macro F1", marker="o", ms=4)
    for lab in LABELS:
        mm = by_epoch(f"eval_val_f1_{lab}")
        if len(mm):
            a_f1.plot(mm.index, mm["mean"], color=CLASS_COLOR[lab], lw=1.6, label=lab)
    a_f1.set(title="Validation F1 per class", xlabel="epoch", ylabel="F1 (argmax)")
    a_f1.legend(loc="lower right")

    for ax, col, ttl in (
        (a_grad, "grad_norm", "Gradient norm (spikes / NaN = unstable)"),
        (a_lr, "learning_rate", "Learning-rate schedule"),
    ):
        d = h.dropna(subset=[col]) if col in h else pd.DataFrame()
        for i, (k, g) in enumerate(d.groupby("fold") if len(d) else []):
            ax.plot(g["epoch"], g[col], color=SERIES[i % len(SERIES)], lw=1.2, label=f"fold {int(k)}")
        ax.set(title=ttl, xlabel="epoch")
        if len(folds) > 1:
            ax.legend()
    a_lr.ticklabel_format(axis="y", style="sci", scilimits=(-3, 3))
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
    _save(fig, path)


def plot_confusion(frames: dict[str, pd.DataFrame], title: str, path: Path) -> None:
    fig, axes = plt.subplots(1, len(frames), figsize=(5.2 * len(frames), 4.4), squeeze=False)
    for ax, (name, df) in zip(axes[0], frames.items()):
        cm = pd.crosstab(
            pd.Categorical(df["label"], LABELS), pd.Categorical(df["pred"], LABELS), dropna=False
        )
        rate = cm.to_numpy() / np.maximum(cm.to_numpy().sum(1, keepdims=True), 1)
        ax.imshow(rate, cmap=BLUES, vmin=0, vmax=1)
        ax.grid(False)
        short = ["BENIGN", "JAILBR.", "P_INJ."]
        ax.set_xticks(range(3), short)
        ax.set_yticks(range(3), short)
        ax.set(
            xlabel="predicted",
            ylabel="true",
            title=f"{name}: {len(df)} rows, macro F1 "
            f"{macro_f1(df['label'].map(LABELS.index).to_numpy(), df['pred'].map(LABELS.index).to_numpy()):.4f}",
        )
        for i in range(3):
            for j in range(3):
                ax.text(
                    j,
                    i,
                    f"{cm.iat[i, j]}\n{rate[i, j]:.1%}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if rate[i, j] > 0.55 else INK,
                )
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
    _save(fig, path)


def _f1_by(df: pd.DataFrame, key: str, order) -> pd.DataFrame:
    from sklearn.metrics import f1_score

    rows = []
    for v in order:
        d = df[df[key] == v]
        if len(d) == 0:
            continue
        y, p = d["label"].map(LABELS.index), d["pred"].map(LABELS.index)
        f = f1_score(y, p, labels=[0, 1, 2], average=None, zero_division=0)
        rows.append(dict(group=v, n=len(d), macro=f.mean(), **dict(zip(LABELS, f))))
    return pd.DataFrame(rows)


def plot_slices(df: pd.DataFrame, title: str, path: Path) -> None:
    """F1 per class by language and by prompt length — where the model breaks."""
    df = df.copy()
    df["length"] = pd.cut(
        df["n_chars"],
        [b[0] for b in LENGTH_BUCKETS] + [10**9],
        right=False,
        labels=[b[2] for b in LENGTH_BUCKETS],
    )
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
    for ax, key, order, ttl in (
        (axes[0], "language", ["pt", "fr", "it", "en"], "F1 by language"),
        (axes[1], "length", [b[2] for b in LENGTH_BUCKETS], "F1 by prompt length (chars)"),
    ):
        t = _f1_by(df, key, order)
        x = np.arange(len(t))
        w = 0.26
        for i, lab in enumerate(LABELS):
            ax.bar(
                x + (i - 1) * w,
                t[lab],
                width=w,
                color=CLASS_COLOR[lab],
                edgecolor=SURFACE,
                linewidth=2,
                label=lab,
            )
        ax.plot(x, t["macro"], color=INK, marker="D", ms=5, lw=0, label="macro F1")
        for xi, (v, n) in enumerate(zip(t["macro"], t["n"])):
            ax.text(xi, 1.005, f"{v:.3f}\nn={n}", ha="center", va="bottom", fontsize=7, color=INK2)
        ax.set_xticks(x, t["group"])
        ax.set_ylim(0, 1.12)
        ax.set(title=ttl, ylabel="F1")
        ax.grid(axis="x", visible=False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", ncol=4, bbox_to_anchor=(1, 1.0))
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
    _save(fig, path)


def plot_confidence(df: pd.DataFrame, title: str, path: Path) -> None:
    """Is the model's confidence honest? Histogram (right vs wrong) + reliability curve."""
    p = df[[f"p_{lab}" for lab in LABELS]].to_numpy()
    conf = p.max(1)
    correct = (df["label"] == np.asarray(LABELS)[p.argmax(1)]).to_numpy()
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.2))
    bins = np.linspace(1 / 3, 1, 21)
    a.hist(conf[correct], bins=bins, color=SERIES[0], label=f"correct ({correct.sum()})", edgecolor=SURFACE)
    a.hist(conf[~correct], bins=bins, color=SERIES[1], label=f"wrong ({(~correct).sum()})", edgecolor=SURFACE)
    a.set_yscale("log")
    a.set(title="Confidence of the top class (log scale)", xlabel="max probability", ylabel="rows")
    a.legend()
    edges = np.linspace(1 / 3, 1, 9)
    idx = np.clip(np.digitize(conf, edges) - 1, 0, len(edges) - 2)
    acc = [correct[idx == i].mean() if (idx == i).any() else np.nan for i in range(len(edges) - 1)]
    mid = [conf[idx == i].mean() if (idx == i).any() else np.nan for i in range(len(edges) - 1)]
    b.plot([1 / 3, 1], [1 / 3, 1], color=MUTED, lw=1, ls="--", label="perfectly calibrated")
    b.plot(mid, acc, color=SERIES[0], marker="o", ms=6, label="model")
    b.set(title="Reliability: accuracy at each confidence level", xlabel="mean confidence", ylabel="accuracy")
    b.legend(loc="upper left")
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
    _save(fig, path)


def top_errors(df: pd.DataFrame, n: int = 60) -> pd.DataFrame:
    p = df[[f"p_{lab}" for lab in LABELS]]
    wrong = df[df["label"] != df["pred"]].copy()
    wrong["confidence"] = p.loc[wrong.index].max(axis=1)
    return wrong.sort_values("confidence", ascending=False).head(n)


def plot_run(run_dir: str | Path) -> list[Path]:
    run_dir = Path(run_dir)
    meta = json.loads((run_dir / "meta.json").read_text())
    name = meta["name"]
    charts = run_dir / "charts"
    made = []
    hist_path = run_dir / "history.csv"
    if hist_path.exists() and hist_path.stat().st_size > 5:
        h = pd.read_csv(hist_path)
        if "eval_val_loss" in h or "loss" in h:
            plot_training_health(h, f"{name} — training health", charts / "training_health.png")
            made.append(charts / "training_health.png")
    preds = load_predictions(run_dir)
    if preds:
        plot_confusion(preds, f"{name} — confusion (row-normalised)", charts / "confusion.png")
        main = preds.get("holdout", preds.get("oof"))
        which = "holdout" if "holdout" in preds else "out-of-fold"
        plot_slices(main, f"{name} — where it breaks ({which})", charts / "slices.png")
        plot_confidence(main, f"{name} — confidence ({which})", charts / "confidence.png")
        top_errors(main).to_csv(run_dir / "top_errors.csv", index=False)
        made += [charts / "confusion.png", charts / "slices.png", charts / "confidence.png"]
    return made


# ------------------------------------------------------------------------------------------- compare
def summarize(run_dir: Path) -> dict:
    meta = json.loads((run_dir / "meta.json").read_text())
    s, ho = meta.get("scores") or {}, meta.get("holdout")
    src = ho or s
    pc = pd.DataFrame(src.get("per_class", {}))
    pl = pd.DataFrame(src.get("per_language", {}))
    folds = meta.get("fold_stats", [])
    row = dict(
        model=meta["name"],
        backbone=meta.get("model"),
        macro_f1=(ho or {}).get("macro_f1", s.get("macro_f1_tuned_crossfit", s.get("macro_f1_tuned"))),
        macro_f1_argmax=src.get("macro_f1_argmax"),
        minutes_per_fold=float(np.mean([f.get("minutes", np.nan) for f in folds])) if folds else np.nan,
        peak_vram_gb=float(np.max([f.get("peak_vram_gb", 0) for f in folds])) if folds else np.nan,
        truncated=float(np.mean([f.get("truncated", np.nan) for f in folds])) if folds else np.nan,
        folds=len(folds),
    )
    for lab in LABELS:
        row[f"f1_{lab}"] = pc.get("f1", {}).get(lab, np.nan) if len(pc) else np.nan
    if len(pl):
        row["pt_f1_JAILBREAK"] = pl.get("f1_JAILBREAK", {}).get("pt", np.nan)
        row["en_f1_PROMPT_INJECTION"] = pl.get("f1_PROMPT_INJECTION", {}).get("en", np.nan)
    h = run_dir / "history.csv"
    if h.exists() and h.stat().st_size > 5:
        hh = pd.read_csv(h)
        if "eval_val_loss" in hh and "eval_train_loss" in hh:
            last = hh["epoch"].max()
            v = hh.loc[hh["epoch"] == last, "eval_val_loss"].dropna()
            t = hh.loc[hh["epoch"] == last, "eval_train_loss"].dropna()
            row["final_val_loss"] = float(v.mean()) if len(v) else np.nan
            row["overfit_gap"] = float(v.mean() - t.mean()) if len(v) and len(t) else np.nan
            row["best_val_f1_epoch"] = (
                float(hh.loc[hh["eval_val_macro_f1"].idxmax(), "epoch"])
                if "eval_val_macro_f1" in hh and hh["eval_val_macro_f1"].notna().any()
                else np.nan
            )
    return row


def plot_compare(
    run_dirs: list[str | Path], out_dir: str | Path, title: str = "Benchmark — fold 0"
) -> pd.DataFrame:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    runs = [Path(d) for d in run_dirs]
    t = (
        pd.DataFrame([summarize(d) for d in runs])
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )
    t.to_csv(out / "summary.csv", index=False)

    # 1. ranking
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.4))
    _bars(ax, t["model"], t["macro_f1"], xlim=_f1_xlim(t["macro_f1"]))
    ax.set(title=f"{title}: macro F1 (decision offsets tuned)", xlabel="macro F1")
    _save(fig, out / "ranking.png")

    # 2. per-class F1 — small multiples, one panel per class
    fig, axes = plt.subplots(1, 3, figsize=(15, 0.42 * len(t) + 1.6), sharey=True)
    for ax, lab in zip(axes, LABELS):
        _bars(ax, t["model"], t[f"f1_{lab}"], color=CLASS_COLOR[lab], xlim=_f1_xlim(t[f"f1_{lab}"]))
        ax.set(title=f"F1 {lab}", xlabel="F1")
    _save(fig, out / "per_class.png")

    # 3. the two bottlenecks found in EDA
    cols = [c for c in ("pt_f1_JAILBREAK", "en_f1_PROMPT_INJECTION") if c in t]
    if cols:
        fig, axes = plt.subplots(
            1, len(cols), figsize=(7.5 * len(cols), 0.42 * len(t) + 1.6), sharey=True, squeeze=False
        )
        names = {
            "pt_f1_JAILBREAK": ("Bottleneck 1: JAILBREAK F1 in Portuguese", CLASS_COLOR["JAILBREAK"]),
            "en_f1_PROMPT_INJECTION": (
                "Bottleneck 2: PROMPT_INJECTION F1 in English",
                CLASS_COLOR["PROMPT_INJECTION"],
            ),
        }
        for ax, c in zip(axes[0], cols):
            _bars(ax, t["model"], t[c], color=names[c][1], xlim=_f1_xlim(t[c]))
            ax.set(title=names[c][0], xlabel="F1")
        _save(fig, out / "bottlenecks.png")

    # 4. cost vs score
    d = t.dropna(subset=["minutes_per_fold", "macro_f1"])
    if len(d):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(
            d["minutes_per_fold"],
            d["macro_f1"],
            s=70,
            color=SERIES[0],
            edgecolor=SURFACE,
            linewidth=2,
            zorder=3,
        )
        for _, r in d.iterrows():
            ax.annotate(
                r["model"],
                (r["minutes_per_fold"], r["macro_f1"]),
                xytext=(6, 4),
                textcoords="offset points",
                fontsize=8,
                color=INK,
            )
        ax.set_xscale("log")
        ax.set(
            title="Cost vs score: minutes per fold (log) against macro F1",
            xlabel="minutes per fold",
            ylabel="macro F1",
        )
        _save(fig, out / "cost_vs_score.png")

    # 5. learning curves — small multiples (one panel per model, same axes)
    hist = {}
    for d_ in runs:
        h = d_ / "history.csv"
        if h.exists() and h.stat().st_size > 5:
            hh = pd.read_csv(h)
            if "eval_val_loss" in hh:
                hist[json.loads((d_ / "meta.json").read_text())["name"]] = hh
    if hist:
        order = [m for m in t["model"] if m in hist]
        ncol = min(4, len(order))
        nrow = int(np.ceil(len(order) / ncol))
        for metric, fname, ylab in (
            ("loss", "learning_curves_loss.png", "loss"),
            ("f1", "learning_curves_f1.png", "validation macro F1"),
        ):
            fig, axes = plt.subplots(
                nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow), sharex=False, sharey=True, squeeze=False
            )
            for ax in axes.flat[len(order) :]:
                ax.set_visible(False)
            for ax, m in zip(axes.flat, order):
                hh = hist[m]
                if metric == "loss":
                    for col, color, lab in (
                        ("eval_train_loss", SERIES[1], "train (probe)"),
                        ("eval_val_loss", SERIES[2], "validation"),
                    ):
                        g = hh.dropna(subset=[col]).groupby("epoch")[col].mean()
                        ax.plot(g.index, g.values, color=color, marker="o", ms=3, label=lab)
                else:
                    g = hh.dropna(subset=["eval_val_macro_f1"]).groupby("epoch")["eval_val_macro_f1"].mean()
                    ax.plot(g.index, g.values, color=SERIES[0], marker="o", ms=3)
                ax.set_title(m, fontsize=9)
                ax.set_xlabel("epoch")
            axes[0][0].set_ylabel(ylab)
            if metric == "loss":
                axes[0][0].legend()
            fig.suptitle(
                f"{title}: {ylab} per epoch (one panel per model)",
                x=0.01,
                ha="left",
                fontsize=13,
                fontweight="bold",
                color=INK,
            )
            _save(fig, out / fname)
    return t

In [ ]:
%%writefile src/rmitguard/cli.py
"""Command line entry point: `rmitguard <command>` (or `python -m rmitguard <command>`)."""

from __future__ import annotations

import argparse
import sys

from .constants import L2I, LABELS, N_TEST_ROWS


def _folds(value: str | None) -> list[int] | None:
    return None if value in (None, "", "all") else [int(x) for x in value.split(",")]


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(prog="rmitguard", description=__doc__)
    parser.add_argument(
        "--data", default=None, help="folder with train.csv / test.csv (default: auto-detect)"
    )
    sub = parser.add_subparsers(dest="cmd", required=True)

    p = sub.add_parser("eda", help="write reports/eda.md")
    p.add_argument("--out", default="reports/eda.md")

    p = sub.add_parser("train", help="run one experiment config with K-fold CV")
    p.add_argument("config")
    p.add_argument("--folds", default=None, help="comma-separated subset, e.g. 0,1 (default: all)")
    p.add_argument("--limit", type=int, default=None, help="train on a small stratified subset (smoke test)")
    p.add_argument("--out", default="artifacts")
    p.add_argument("--name", default=None, help="override the experiment name")
    p.add_argument("--holdout", type=float, default=None, help="lock this share of train away, e.g. 0.2")

    p = sub.add_parser("ensemble", help="merge fold subsets / runs that share a CV split")
    p.add_argument("runs", nargs="+", help="run directories or roots to search for them")
    p.add_argument("--out", default="artifacts/ensemble")
    p.add_argument("--weights", default=None, help="name=weight,name=weight")
    p.add_argument("--allow-multi-backbone", action="store_true")
    p.add_argument("--only", default=None, help="comma-separated run names to keep, e.g. mmbert-base")

    p = sub.add_parser("validate", help="check a submission CSV before spending a submission on it")
    p.add_argument("submission")

    p = sub.add_parser("xling", help="cross-lingual nearest neighbours with pre-trained mmBERT-base")
    p.add_argument("--out", default="artifacts/xling")

    sub.add_parser("labels", help="print the label order used everywhere")

    args = parser.parse_args(argv)

    if args.cmd == "eda":
        from .eda import eda_report

        print(eda_report(args.data, args.out))
    elif args.cmd == "train":
        from .config import load_config
        from .runner import run_experiment

        cfg = load_config(args.config, folds=_folds(args.folds), name=args.name, holdout=args.holdout)
        run_experiment(cfg, data_dir=args.data, out_dir=args.out, limit=args.limit)
    elif args.cmd == "ensemble":
        from .ensemble import ensemble

        weights = (
            dict((k, float(v)) for k, v in (kv.split("=") for kv in args.weights.split(",")))
            if args.weights
            else None
        )
        ensemble(
            args.runs,
            out_dir=args.out,
            data_dir=args.data,
            weights=weights,
            allow_multi_backbone=args.allow_multi_backbone,
            only=args.only.split(",") if args.only else None,
        )
    elif args.cmd == "validate":
        import pandas as pd

        from .data import find_data_dir
        from .submission import label_distribution, validate_submission

        try:
            test_ids = pd.read_csv(find_data_dir(args.data) / "test.csv")["id"]
        except FileNotFoundError:
            test_ids = None
            print("⚠️ test.csv not found; checking format only")
        problems = validate_submission(
            args.submission, test_ids, expected_rows=len(test_ids) if test_ids is not None else N_TEST_ROWS
        )
        if problems:
            print("❌ NOT safe to submit:\n- " + "\n- ".join(problems))
            return 1
        print(
            "✅ valid —", {k: f"{v:.1%}" for k, v in label_distribution(pd.read_csv(args.submission)).items()}
        )
    elif args.cmd == "xling":
        from .xling import run_xling

        run_xling(args.data, args.out)
    elif args.cmd == "labels":
        print({lab: L2I[lab] for lab in LABELS})
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile src/rmitguard/config.py
"""Experiment configuration: one YAML file in configs/ = one experiment."""

from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import yaml

from .constants import APPROVED_MODELS


@dataclass
class ExperimentConfig:
    name: str
    model: str  # key of APPROVED_MODELS, or "tfidf" (local diagnostic only, not an approved backbone)
    seed: int = 42  # training seed: change it for extra ensemble members
    split_seed: int = 42  # CV split: keep it fixed so every run of the team can be merged
    n_folds: int = 5
    folds: list[int] | None = None  # None = all folds; a subset lets teammates split the work
    holdout: float = 0.0  # 0.2 = lock 20% of train away as a private-LB stand-in (benchmark protocol)
    # Tokenisation: long prompts keep head + tail (role-play set-up at the start, real request at the end).
    max_len: int = 512
    head_len: int = 384
    prepend_language: bool = False
    # Optimisation
    lr: float = 2e-5
    epochs: int = 3
    batch_size: int = 16
    grad_accum: int = 1
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    class_weight: str = "sqrt"  # none | sqrt | balanced
    label_smoothing: float = 0.0
    grad_ckpt: bool = False
    # Memory: DataParallel keeps weights + optimizer + the gradient reduction on GPU 0, which overflows a T4
    # for xlmr-large. n_gpu: 1 trains on one GPU; freeze_embeddings drops the 250k-token embedding matrix
    # (~half of the parameters of XLM-R / mmBERT) from the optimizer.
    n_gpu: int | None = None
    freeze_embeddings: bool = False
    fp16: bool = True
    # LLM route only (kind == "llm"): 4-bit base + LoRA adapters + classification head
    lora: dict[str, Any] = field(default_factory=lambda: {"r": 16, "alpha": 32, "dropout": 0.05})
    # TF-IDF only
    tfidf: dict[str, Any] = field(default_factory=lambda: {"C": 8.0})
    # Stop starting new folds when the next one would not finish inside Kaggle's 12 h limit.
    time_budget_hours: float = 11.0
    # Training-health logging: evaluations per epoch on the validation fold and on a fixed train probe
    # (same loss computation on both, so train vs eval loss is directly comparable -> over/under-fitting).
    evals_per_epoch: int = 4
    train_probe: int = 512
    notes: str = ""

    def __post_init__(self) -> None:
        if self.model != "tfidf" and self.model not in APPROVED_MODELS:
            raise ValueError(
                f"model {self.model!r} is not approved; choose from {sorted(APPROVED_MODELS)} or 'tfidf'"
            )
        if self.class_weight not in {"none", "sqrt", "balanced"}:
            raise ValueError(f"class_weight must be none|sqrt|balanced, got {self.class_weight!r}")
        if self.folds is not None and any(not 0 <= k < self.n_folds for k in self.folds):
            raise ValueError(f"folds {self.folds} out of range for n_folds={self.n_folds}")
        if not 0 <= self.holdout < 0.5:
            raise ValueError("holdout must be in [0, 0.5)")
        if not 0 < self.head_len <= self.max_len:
            raise ValueError("need 0 < head_len <= max_len")

    @property
    def kind(self) -> str:
        return "tfidf" if self.model == "tfidf" else APPROVED_MODELS[self.model][2]

    @property
    def hf_id(self) -> str | None:
        return APPROVED_MODELS[self.model][0] if self.model in APPROVED_MODELS else None

    @property
    def kaggle_model(self) -> str | None:
        return APPROVED_MODELS[self.model][1] if self.model in APPROVED_MODELS else None

    def fold_ids(self) -> list[int]:
        return list(range(self.n_folds)) if self.folds is None else sorted(self.folds)

    def to_dict(self) -> dict[str, Any]:
        return dataclasses.asdict(self)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> ExperimentConfig:
        known = {f.name for f in dataclasses.fields(cls)}
        unknown = sorted(set(data) - known)
        if unknown:
            raise ValueError(f"unknown config keys {unknown} (typo?)")
        return cls(**data)


def load_config(path: str | Path, **overrides: Any) -> ExperimentConfig:
    data = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
    data.update({k: v for k, v in overrides.items() if v is not None})
    return ExperimentConfig.from_dict(data)


def load_config_text(text: str, **overrides: Any) -> ExperimentConfig:
    data = yaml.safe_load(text) or {}
    data.update({k: v for k, v in overrides.items() if v is not None})
    return ExperimentConfig.from_dict(data)

In [ ]:
%%writefile src/rmitguard/constants.py
"""Competition facts. Everything here comes from the competition pages (fetched 2026-10-21)."""

COMPETITION = "rmit-hackathon-2026"
DEADLINE_UTC = "2026-10-22T11:00:00Z"  # 18:00 Vietnam time
MAX_DAILY_SUBMISSIONS = 6

LABELS = ("BENIGN", "JAILBREAK", "PROMPT_INJECTION")
L2I = {label: i for i, label in enumerate(LABELS)}
LANGUAGES = ("pt", "fr", "it", "en")
N_TEST_ROWS = 6000

# The ONLY backbones the rules allow. key -> (Hugging Face id, Kaggle model handle, kind)
# Kaggle handles are attached to a notebook via `model_sources` and mounted under /kaggle/input.
APPROVED_MODELS = {
    "distilbert-mbert": (
        "distilbert/distilbert-base-multilingual-cased",
        "aisuko/distilbert-base-multilingual-cased/Transformers/default/1",
        "encoder",
    ),
    "mmbert-small": ("jhu-clsp/mmBERT-small", "aisuko/mmbert-small/Transformers/default/1", "encoder"),
    "mmbert-base": ("jhu-clsp/mmBERT-base", "aisuko/mmbert-base/Transformers/default/1", "encoder"),
    "xlmr-base": ("FacebookAI/xlm-roberta-base", "aisuko/xlm-roberta-base/Transformers/default/1", "encoder"),
    "xlmr-large": (
        "FacebookAI/xlm-roberta-large",
        "aisuko/xlm-roberta-large/Transformers/default/1",
        "encoder",
    ),
    "rembert": ("google/rembert", "aisuko/rembert/Transformers/default/1", "encoder"),
    "deberta-pi-v2": (
        "protectai/deberta-v3-base-prompt-injection-v2",
        "aisuko/deberta-v3-base-prompt-injection-v2/Transformers/default/1",
        "encoder",
    ),
    "wolf-defender": (
        "patronus-studio/wolf-defender-prompt-injection",
        "aisuko/wolf-defender-prompt-injection/Transformers/default/1",
        "encoder",
    ),
    "granite-2b": (
        "ibm-granite/granite-3.3-2b-instruct",
        "aisuko/granite-3.3-2b-instruct/Transformers/default/1",
        "llm",
    ),
    "mistral-7b": (
        "mistralai/Mistral-7B-Instruct-v0.3",
        "aisuko/mistral-7b-instruct-v0.3/Transformers/default/1",
        "llm",
    ),
}

In [ ]:
%%writefile src/rmitguard/data.py
"""Loading the competition CSVs and building leak-free, reproducible CV folds."""

from __future__ import annotations

import glob
import hashlib
import os
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from .constants import COMPETITION, L2I, LABELS

KAGGLE_DATA = f"/kaggle/input/competitions/{COMPETITION}"


def find_data_dir(explicit: str | Path | None = None) -> Path:
    """Explicit path > $RMITGUARD_DATA > ./data > Kaggle mount (any depth, model folders excluded)."""
    candidates = [explicit, os.environ.get("RMITGUARD_DATA"), "data", KAGGLE_DATA]
    for c in candidates:
        if c and (Path(c) / "train.csv").exists():
            return Path(c)
    hits = [p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True) if "/models/" not in p]
    hits.sort(key=lambda p: ("rmit" not in p.lower(), p))
    if hits:
        return Path(hits[0]).parent
    raise FileNotFoundError(
        "train.csv not found. Locally: `make data` (you must have joined the competition on Kaggle). "
        "On Kaggle: Add Input -> Competitions -> RMIT HACKATHON 2026."
    )


@dataclass
class CompetitionData:
    train: pd.DataFrame
    test: pd.DataFrame

    @property
    def y(self) -> np.ndarray:
        return self.train["label"].map(L2I).to_numpy()


def load_competition(data_dir: str | Path | None = None) -> CompetitionData:
    d = find_data_dir(data_dir)
    train, test = pd.read_csv(d / "train.csv"), pd.read_csv(d / "test.csv")
    for df in (train, test):
        df["text"] = df["text"].fillna("").astype(str)
        if "language" not in df:
            df["language"] = "unk"
    bad = sorted(set(train["label"]) - set(LABELS))
    if bad:
        raise ValueError(f"unexpected labels in train.csv: {bad}")
    return CompetitionData(train=train, test=test)


def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\s+", " ", text).strip()


def text_hash(text: str) -> str:
    return hashlib.md5(normalize(text).encode("utf-8")).hexdigest()


def make_folds(train: pd.DataFrame, n_folds: int, seed: int) -> np.ndarray:
    """Fold id per row.

    Stratified on label x language so every fold sees the rare PROMPT_INJECTION class in every language,
    and grouped on the normalised text so duplicated prompts never sit on both sides of a split
    (duplicates across folds inflate CV and make it useless for picking models).

    Implemented with numpy's legacy RandomState, whose stream is frozen across numpy versions, so the split
    is identical on a laptop and on Kaggle (sklearn's StratifiedGroupKFold gave different folds per version).
    """
    strata = (train["label"].astype(str) + "|" + train["language"].astype(str)).to_numpy()
    groups = train["text"].map(text_hash).to_numpy()
    g = pd.DataFrame({"group": groups, "stratum": strata}).drop_duplicates("group")
    rng = np.random.RandomState(seed)
    g = g.iloc[rng.permutation(len(g))]
    offsets = {s: int(rng.randint(n_folds)) for s in sorted(g["stratum"].unique())}
    g["fold"] = (g.groupby("stratum").cumcount() + g["stratum"].map(offsets)) % n_folds
    fold_of = dict(zip(g["group"], g["fold"]))
    return np.array([fold_of[h] for h in groups], dtype=int)


def make_split(train: pd.DataFrame, n_folds: int, seed: int, holdout: float = 0.0) -> np.ndarray:
    """Fold id per row; -1 marks the locked holdout (stratified + grouped the same way as the folds)."""
    if not holdout:
        return make_folds(train, n_folds, seed)
    is_hold = make_folds(train, round(1 / holdout), seed + 1) == 0
    folds = np.full(len(train), -1, dtype=int)
    folds[~is_hold] = make_folds(train[~is_hold].reset_index(drop=True), n_folds, seed)
    return folds


def split_fingerprint(folds: np.ndarray) -> str:
    """Runs can only be merged (ensembles, fold subsets from teammates) when this matches."""
    return hashlib.md5(folds.astype(np.int64).tobytes()).hexdigest()[:12]

In [ ]:
%%writefile src/rmitguard/eda.py
"""Quick EDA that answers the questions that drive modelling decisions (writes a Markdown report)."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .data import load_competition, text_hash


def eda_report(data_dir: str | None = None, out: str | Path = "reports/eda.md") -> str:
    data = load_competition(data_dir)
    train, test = data.train.copy(), data.test.copy()
    for df in (train, test):
        df["n_chars"] = df["text"].str.len()
        df["n_words"] = df["text"].str.split().str.len()
        df["hash"] = df["text"].map(text_hash)

    dup = train.groupby("hash")["label"].agg(["count", "nunique"])
    test_in_train = test["hash"].isin(set(train["hash"]))
    leaked = test[test_in_train].merge(train[["hash", "label"]].drop_duplicates("hash"), on="hash")
    q = [0.5, 0.9, 0.99]
    lines = [
        "# EDA — RMIT Hackathon 2026 (Challenge 1)",
        "",
        f"train {train.shape} · test {test.shape}",
        "",
        "## Label × language (train, counts)",
        pd.crosstab(train["language"], train["label"], margins=True).to_markdown(),
        "",
        "## Label share per language (row %)",
        (pd.crosstab(train["language"], train["label"], normalize="index") * 100).round(1).to_markdown(),
        "",
        "## Language share: train vs test (%)",
        pd.DataFrame(
            {
                "train": train["language"].value_counts(normalize=True) * 100,
                "test": test["language"].value_counts(normalize=True) * 100,
            }
        )
        .round(1)
        .to_markdown(),
        "",
        "## Length in characters (quantiles 50/90/99) — decides max_len",
        train.groupby("label")["n_chars"].quantile(q).unstack().round(0).to_markdown(),
        "",
        f"test: {test['n_chars'].quantile(q).round(0).to_dict()}",
        "",
        "## Duplicates (after lower-case + whitespace normalisation)",
        f"- duplicated groups in train: {(dup['count'] > 1).sum()} "
        f"(rows involved: {int(dup.loc[dup['count'] > 1, 'count'].sum())})",
        f"- groups with CONFLICTING labels: {(dup['nunique'] > 1).sum()}",
        f"- test prompts identical to a train prompt: {int(test_in_train.sum())} "
        f"({test_in_train.mean():.1%}) → label distribution of those: "
        f"{leaked['label'].value_counts().to_dict()}",
        "",
        "## Random examples per label",
    ]
    rng = np.random.default_rng(0)
    for label, g in train.groupby("label"):
        lines.append(f"### {label}")
        for _, row in g.iloc[rng.choice(len(g), min(4, len(g)), replace=False)].iterrows():
            snippet = row["text"][:300].replace("\n", " ").replace("|", "/")
            lines.append(f"- `{row['language']}` {snippet}{'…' if len(row['text']) > 300 else ''}")
        lines.append("")
    text = "\n".join(lines) + "\n"
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    Path(out).write_text(text, encoding="utf-8")
    return text

In [ ]:
%%writefile src/rmitguard/ensemble.py
"""Merge run directories: fold subsets run by different teammates, and/or several runs (seeds, configs).

Rules enforced here:
- every run must share the same split fingerprint (same folds -> OOF predictions are comparable);
- the competition allows ONE backbone, so mixing backbones needs an explicit --allow-multi-backbone.
"""

from __future__ import annotations

import json
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from .data import load_competition
from .postprocess import predict_with_bias
from .runner import _jsonable, predictions_frame, score_holdout, score_oof, write_report
from .submission import make_submission, validate_submission


def find_runs(roots: list[str | Path]) -> list[Path]:
    """Every directory below the given roots that contains meta.json + oof.npy + test.npy."""
    found = []
    for root in roots:
        root = Path(root)
        metas = [root / "meta.json"] if (root / "meta.json").exists() else sorted(root.rglob("meta.json"))
        found += [
            m.parent for m in metas if (m.parent / "oof.npy").exists() and (m.parent / "test.npy").exists()
        ]
    return sorted(set(found))


def load_run(run_dir: Path) -> dict:
    meta = json.loads((run_dir / "meta.json").read_text(encoding="utf-8"))
    hold = run_dir / "holdout.npy"
    return dict(
        dir=run_dir,
        meta=meta,
        oof=np.load(run_dir / "oof.npy"),
        test=np.load(run_dir / "test.npy"),
        holdout=np.load(hold) if hold.exists() else None,
    )


def merge_fold_subsets(runs: list[dict]) -> dict[str, dict]:
    """Same experiment name, different folds -> one run with the union of folds."""
    by_name = defaultdict(list)
    for r in runs:
        by_name[r["meta"]["name"]].append(r)
    merged = {}
    for name, parts in by_name.items():
        oof = np.full_like(parts[0]["oof"], np.nan)
        test_sum, hold_sum, n_models, folds_done, model = 0.0, 0.0, 0, [], parts[0]["meta"]["model"]
        for p in parts:
            overlap = sorted(set(p["meta"]["folds_done"]) & set(folds_done))
            if overlap:
                print(f"⚠️ {name}: fold(s) {overlap} present twice ({p['dir']}); keeping the first copy")
                continue
            m = ~np.isnan(p["oof"]).any(axis=1)
            oof[m] = p["oof"][m]
            test_sum = test_sum + p["test"] * p["meta"]["n_test_models"]
            if p["holdout"] is not None:
                hold_sum = hold_sum + p["holdout"] * p["meta"]["n_test_models"]
            n_models += p["meta"]["n_test_models"]
            folds_done += p["meta"]["folds_done"]
        merged[name] = dict(
            oof=oof,
            test=test_sum / n_models,
            holdout=None if parts[0]["holdout"] is None else hold_sum / n_models,
            folds_done=sorted(folds_done),
            model=model,
            sources=[str(p["dir"]) for p in parts],
        )
    return merged


def ensemble(
    roots: list[str | Path],
    out_dir: str | Path = "artifacts/ensemble",
    data_dir: str | None = None,
    weights: dict[str, float] | None = None,
    allow_multi_backbone: bool = False,
    only: list[str] | None = None,
    xling: str | Path | None = "auto",
    xling_threshold: float | None = None,
) -> Path:
    runs = [
        r for r in (load_run(d) for d in find_runs(roots)) if "members" not in r["meta"]
    ]  # skip old ensembles
    if only:  # e.g. only the B* runs out of a benchmark notebook that trained several models
        runs = [r for r in runs if r["meta"]["name"] in only]
    if not runs:
        raise FileNotFoundError(f"no run directories under {roots}")
    fps = {r["meta"]["split_fp"] for r in runs}
    if len(fps) > 1:
        raise ValueError(
            f"runs use different CV splits {fps}; they cannot be merged (same seed + n_folds needed)"
        )
    merged = merge_fold_subsets(runs)
    backbones = {m["model"] for m in merged.values()}
    if len(backbones) > 1 and not allow_multi_backbone:
        raise ValueError(
            f"runs use several backbones {backbones}; the rules allow ONE approved model. "
            "Pass --allow-multi-backbone only if the organisers confirm it is allowed."
        )

    data = load_competition(data_dir)
    train, test, y = data.train, data.test, data.y
    if len(train) != len(next(iter(merged.values()))["oof"]):
        raise ValueError("runs were made on a different (sub)set of train.csv (e.g. --limit)")
    folds = np.load(runs[0]["dir"] / "folds.npy")
    w = {name: (weights or {}).get(name, 1.0) for name in merged}
    total = sum(w.values())
    oof = sum(w[n] * m["oof"] for n, m in merged.items()) / total  # NaN wherever any member lacks a fold
    test_p = sum(w[n] * m["test"] for n, m in merged.items()) / total

    language = train["language"].astype(str).to_numpy()
    xling_info = None
    if xling == "auto":  # an xling.npz next to the runs (e.g. from a Kaggle input) is picked up automatically
        hits = [p for r in roots for p in Path(r).rglob("xling.npz")] if roots else []
        xling = hits[0] if hits else None
    if xling:
        from .xling import THRESHOLD, apply_override

        z = np.load(xling)
        thr = xling_threshold if xling_threshold is not None else THRESHOLD
        m = ~np.isnan(oof).any(axis=1)
        before = score_oof(oof, y, folds, language)
        if np.array_equal(z["folds"], folds):  # train neighbours come from other folds -> OOF stays honest
            oof = oof.copy()
            oof[m] = apply_override(oof[m], z["train_sim"][m], y[z["train_nn"][m]], thr)
        test_p = apply_override(test_p, z["test_sim"], y[z["test_nn"]], thr)
        xling_info = dict(
            threshold=thr,
            oof_overridden=int((z["train_sim"][m] >= thr).sum()),
            test_overridden=int((z["test_sim"] >= thr).sum()),
            macro_f1_tuned_before=before.get("macro_f1_tuned"),
            macro_f1_crossfit_before=before.get("macro_f1_tuned_crossfit"),
        )
        print(f"xling override (sim >= {thr}): {xling_info}")
    scores = score_oof(oof, y, folds, language)
    bias = np.asarray(scores["bias"])
    hold = folds == -1
    holdout = None
    if hold.any():
        hold_p = sum(w[n] * m["holdout"] for n, m in merged.items()) / total
        holdout = score_holdout(hold_p, y[hold], language[hold], bias)
    sub = make_submission(test["id"], predict_with_bias(test_p, bias))
    problems = validate_submission(sub, test["id"], expected_rows=len(test))
    if problems:
        raise RuntimeError(f"invalid submission: {problems}")

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "oof.npy", oof.astype(np.float32))
    np.save(out / "test.npy", test_p.astype(np.float32))
    np.save(out / "folds.npy", folds)
    if holdout is not None:
        np.save(out / "holdout.npy", hold_p.astype(np.float32))
        predictions_frame(train[hold], hold_p, predict_with_bias(hold_p, bias)).to_csv(
            out / "holdout_predictions.csv", index=False
        )
    m = ~np.isnan(oof).any(axis=1)
    predictions_frame(train[m], oof[m], predict_with_bias(oof[m], bias)).to_csv(
        out / "oof_predictions.csv", index=False
    )
    sub.to_csv(out / "submission.csv", index=False)
    members = {
        n: dict(weight=w[n], folds_done=m["folds_done"], model=m["model"], sources=m["sources"])
        for n, m in merged.items()
    }
    (out / "meta.json").write_text(
        json.dumps(
            dict(
                name=out.name,
                model="+".join(sorted(backbones)),
                members=members,
                split_fp=fps.pop(),
                n_train=len(train),
                n_test=len(test),
                scores=_jsonable(scores),
                holdout=_jsonable(holdout),
                xling=xling_info,
                updated_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
            ),
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    write_report(
        out,
        f"ensemble: {out.name}",
        [f"`{n}` · weight {m['weight']} · folds {m['folds_done']} · {m['model']}" for n, m in members.items()]
        + ([f"cross-lingual override: {xling_info}"] if xling_info else []),
        scores,
        sub,
        [],
        holdout,
    )
    print((out / "report.md").read_text(encoding="utf-8"))
    return out

In [ ]:
%%writefile src/rmitguard/metrics.py
"""Competition metric (macro F1) and the breakdowns we use to decide what to try next."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

from .constants import LABELS


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0))


def per_class_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(LABELS))), zero_division=0
    )
    return pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s}, index=list(LABELS))


def per_language_table(y_true: np.ndarray, y_pred: np.ndarray, language: np.ndarray) -> pd.DataFrame:
    rows = []
    for lang in sorted(set(language)):
        m = language == lang
        rows.append(
            dict(
                language=lang,
                n=int(m.sum()),
                macro_f1=macro_f1(y_true[m], y_pred[m]),
                **{
                    f"f1_{lab}": f
                    for lab, f in zip(
                        LABELS,
                        f1_score(
                            y_true[m],
                            y_pred[m],
                            labels=list(range(len(LABELS))),
                            average=None,
                            zero_division=0,
                        ),
                    )
                },
            )
        )
    return pd.DataFrame(rows).set_index("language")


def confusion_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))
    return pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS])


def bootstrap_ci(
    y_true: np.ndarray, y_pred: np.ndarray, n: int = 1000, seed: int = 0, sample_size: int | None = None
) -> tuple[float, float]:
    """95% CI of macro F1. Pass sample_size=1800 to see how noisy the 30% public leaderboard is."""
    rng = np.random.default_rng(seed)
    size = sample_size or len(y_true)
    scores = [
        macro_f1(y_true[idx], y_pred[idx]) for idx in (rng.integers(0, len(y_true), size) for _ in range(n))
    ]
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

In [ ]:
%%writefile src/rmitguard/models/__init__.py
"""Every model exposes the same function:

    train_fold(cfg, train_df, y_train, val_df, y_val, test_df, seed) -> (val_probs, test_probs, info)

Probabilities are float arrays of shape (n_rows, 3) in the order of constants.LABELS.
"""

from __future__ import annotations

from collections.abc import Callable


def get_trainer(kind: str) -> Callable:
    if kind == "tfidf":
        from .tfidf import train_fold
    else:  # torch / transformers are imported only when actually needed
        from .transformer import train_fold
    return train_fold

In [ ]:
%%writefile src/rmitguard/models/tfidf.py
"""TF-IDF + logistic regression: a 2-minute CPU sanity check of the whole pipeline and a floor for CV.

Not an approved backbone — use it to debug and to compare against, never as the submitted model.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from ..config import ExperimentConfig


def _inputs(cfg: ExperimentConfig, df: pd.DataFrame) -> list[str]:
    if cfg.prepend_language:
        return (df["language"].astype(str).map(lambda x: f"__lang_{x} ") + df["text"]).tolist()
    return df["text"].tolist()


def train_fold(
    cfg: ExperimentConfig,
    train_df: pd.DataFrame,
    y_train: np.ndarray,
    val_df: pd.DataFrame,
    y_val: np.ndarray,
    test_df: pd.DataFrame,
    seed: int,
):
    # char n-grams catch obfuscated / misspelled attack phrases in every language without a tokenizer
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
    char = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), min_df=2, max_features=300_000, sublinear_tf=True
    )
    tr = _inputs(cfg, train_df)
    X_tr = hstack([word.fit_transform(tr), char.fit_transform(tr)]).tocsr()

    def transform(df: pd.DataFrame):
        t = _inputs(cfg, df)
        return hstack([word.transform(t), char.transform(t)]).tocsr()

    class_weight = None if cfg.class_weight == "none" else "balanced"
    clf = LogisticRegression(
        C=cfg.tfidf.get("C", 8.0), max_iter=3000, class_weight=class_weight, random_state=seed
    )
    clf.fit(X_tr, y_train)
    return (
        clf.predict_proba(transform(val_df)),
        clf.predict_proba(transform(test_df)),
        {"n_features": X_tr.shape[1]},
    )

In [ ]:
%%writefile src/rmitguard/models/transformer.py
"""Fine-tune one approved backbone as a 3-way sequence classifier.

- kind == "encoder": full fine-tuning, fp16, both T4s through DataParallel.
- kind == "llm": 4-bit NF4 base + LoRA on all linear layers + a classification head, one GPU
  (the same recipe as the Qwen2.5-3B classifier that was our best single model in 2025).
"""

from __future__ import annotations

import gc
import glob
import inspect
import math
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from datasets import Dataset
from packaging.version import Version
from sklearn.metrics import f1_score
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from transformers.modeling_outputs import SequenceClassifierOutput

from ..config import ExperimentConfig
from ..constants import L2I, LABELS

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
ID2LABEL = dict(enumerate(LABELS))
DTYPE_KW = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
TRAINING_ARG_NAMES = set(inspect.signature(TrainingArguments.__init__).parameters)
# Batches of similar length = far less padding. The argument was renamed in transformers 5.
LENGTH_GROUPING = (
    {"train_sampling_strategy": "group_by_length"}
    if "train_sampling_strategy" in TRAINING_ARG_NAMES
    else {"group_by_length": True}
)


def resolve_model_path(cfg: ExperimentConfig) -> str:
    """Kaggle mounts approved models under /kaggle/input/...; locally fall back to the Hugging Face id."""
    slug = cfg.kaggle_model.split("/")[1]
    hits = [
        os.path.dirname(p)
        for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
        if f"/{slug}/" in p.lower()
    ]
    if hits:
        return sorted(hits, key=len)[0]
    if os.path.isdir("/kaggle/input"):
        print(f"⚠️ {cfg.kaggle_model} is not attached as an input; trying the Hub ({cfg.hf_id}).")
    return cfg.hf_id


# ---------------------------------------------------------------- tokenisation
def _special_token_wrap(tokenizer) -> tuple[list[int], list[int]]:
    probe = tokenizer("x", add_special_tokens=False)["input_ids"]
    full = tokenizer("x", add_special_tokens=True)["input_ids"]
    for i in range(len(full) - len(probe) + 1):
        if full[i : i + len(probe)] == probe:
            return full[:i], full[i + len(probe) :]
    raise ValueError(f"cannot locate special tokens of {tokenizer.__class__.__name__}: {full}")


def head_tail(ids: list[int], body_len: int, head_len: int) -> list[int]:
    if len(ids) <= body_len:
        return ids
    return ids[:head_len] + ids[len(ids) - (body_len - head_len) :]


def encode(tokenizer, df: pd.DataFrame, cfg: ExperimentConfig) -> list[list[int]]:
    texts = df["text"].tolist()
    if cfg.prepend_language:
        texts = [f"[{lang}] {t}" for lang, t in zip(df["language"], texts)]
    prefix, suffix = _special_token_wrap(tokenizer)
    body_len = cfg.max_len - len(prefix) - len(suffix)
    head_len = min(cfg.head_len, body_len)
    ids = tokenizer(texts, add_special_tokens=False, truncation=False)["input_ids"]
    return [prefix + head_tail(x, body_len, head_len) + suffix for x in ids]


def to_dataset(ids: list[list[int]], labels: np.ndarray | None = None) -> Dataset:
    data = {"input_ids": ids, "attention_mask": [[1] * len(x) for x in ids]}
    if labels is not None:
        data["labels"] = [int(v) for v in labels]
    return Dataset.from_dict(data)


# ---------------------------------------------------------------- models
class LastTokenClassifier(torch.nn.Module):
    """Classification head on the last non-pad token, for LLMs without a *ForSequenceClassification class."""

    is_parallelizable = True  # tells Trainer not to move / DataParallel-wrap the 4-bit model
    model_parallel = True

    def __init__(self, base, hidden_size: int, num_labels: int):
        super().__init__()
        self.base = base
        self.config = base.config
        self.score = torch.nn.Linear(hidden_size, num_labels, bias=False).to(base.device)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        hidden = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last = attention_mask.sum(dim=1) - 1  # right padding
        pooled = hidden[torch.arange(hidden.size(0), device=hidden.device), last]
        return SequenceClassifierOutput(logits=self.score(pooled.float()))


def load_encoder(path: str):
    config = AutoConfig.from_pretrained(path, num_labels=len(LABELS), id2label=ID2LABEL, label2id=L2I)
    if hasattr(config, "reference_compile"):  # ModernBERT / mmBERT: torch.compile breaks on T4 + DataParallel
        config.reference_compile = False
    # ignore_mismatched_sizes: the safety-tuned checkpoints ship a 2-class head; it is re-initialised for 3 classes
    model = AutoModelForSequenceClassification.from_pretrained(
        path, config=config, ignore_mismatched_sizes=True
    )
    return model.float()


def load_llm(path: str, cfg: ExperimentConfig, tokenizer):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import BitsAndBytesConfig

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    common = {"quantization_config": bnb, DTYPE_KW: torch.float16, "device_map": {"": 0}}
    lora_kw = dict(
        r=cfg.lora["r"],
        lora_alpha=cfg.lora["alpha"],
        lora_dropout=cfg.lora["dropout"],
        target_modules="all-linear",
    )
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            path, num_labels=len(LABELS), id2label=ID2LABEL, label2id=L2I, **common
        )
        model.config.pad_token_id = tokenizer.pad_token_id
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        model = get_peft_model(model, LoraConfig(task_type="SEQ_CLS", **lora_kw))
    except ValueError as exc:  # architecture has no *ForSequenceClassification
        print(f"No sequence-classification class ({exc.__class__.__name__}); using last-token head.")
        base = AutoModel.from_pretrained(path, **common)
        base.config.use_cache = False
        base = prepare_model_for_kbit_training(
            base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        base = get_peft_model(base, LoraConfig(task_type="FEATURE_EXTRACTION", **lora_kw))
        model = LastTokenClassifier(base, base.config.hidden_size, len(LABELS))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable parameters: {trainable / 1e6:.1f}M")
    return model


# ---------------------------------------------------------------- training
def class_weights(y: np.ndarray, mode: str) -> torch.Tensor:
    counts = np.bincount(y, minlength=len(LABELS)).astype(float)
    w = np.ones(len(LABELS)) if mode == "none" else len(y) / (len(LABELS) * np.maximum(counts, 1))
    if mode == "sqrt":
        w = np.sqrt(w)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


class HealthCallback(TrainerCallback):
    """Adds wall-clock minutes and allocated VRAM to every log row (loss-vs-time and memory charts)."""

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.log_history:
            state.log_history[-1]["elapsed_min"] = round((time.time() - self.t0) / 60, 3)
            if torch.cuda.is_available():
                state.log_history[-1]["vram_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 3)


def eval_metrics(p) -> dict:
    pred = _softmax(p.predictions).argmax(1)
    f1 = f1_score(p.label_ids, pred, labels=list(range(len(LABELS))), average=None, zero_division=0)
    return {"macro_f1": float(f1.mean()), **{f"f1_{lab}": float(v) for lab, v in zip(LABELS, f1)}}


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor, label_smoothing: float = 0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing
        # our loss is a per-batch mean, so let Trainer divide by gradient_accumulation_steps
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        loss = F.cross_entropy(
            logits.float(),
            labels,
            weight=self.class_weights.to(logits.device),
            label_smoothing=self.label_smoothing,
        )
        return (loss, outputs) if return_outputs else loss


def make_training_args(**kwargs) -> TrainingArguments:
    unsupported = sorted(k for k in kwargs if k not in TRAINING_ARG_NAMES)
    if unsupported:
        print(f"⚠️ transformers {transformers.__version__} does not support {unsupported}; ignoring them.")
    return TrainingArguments(**{k: v for k, v in kwargs.items() if k in TRAINING_ARG_NAMES})


def _softmax(logits) -> np.ndarray:
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    return torch.softmax(torch.tensor(np.asarray(logits), dtype=torch.float32), dim=-1).numpy()


@torch.no_grad()
def predict_probs(model, tokenizer, ids: list[list[int]], batch_size: int, fp16: bool) -> np.ndarray:
    """Own inference loop, row order guaranteed.

    Trainer.predict in transformers 5 returned rows in a different order than the dataset (its label_ids no
    longer matched the input), so the first xlmr-large run scored 0.35 OOF while its in-training eval said 0.94.
    Batches are formed in length order (much less padding) and written back to their original rows.
    """
    model.eval()
    device = next(p for p in model.parameters()).device
    collate = DataCollatorWithPadding(tokenizer)
    order = np.argsort([len(x) for x in ids], kind="stable")
    out = np.empty((len(ids), len(LABELS)), dtype=np.float32)
    use_amp = fp16 and device.type == "cuda"
    for start in range(0, len(order), batch_size):
        idx = order[start : start + batch_size]
        batch = collate([{"input_ids": ids[i], "attention_mask": [1] * len(ids[i])} for i in idx])
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits
        out[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    return out


def train_fold(
    cfg: ExperimentConfig,
    train_df: pd.DataFrame,
    y_train: np.ndarray,
    val_df: pd.DataFrame,
    y_val: np.ndarray,
    test_df: pd.DataFrame,
    seed: int,
):
    t0 = time.time()
    is_llm = cfg.kind == "llm"
    path = resolve_model_path(cfg)
    tokenizer = AutoTokenizer.from_pretrained(path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if is_llm:
        tokenizer.padding_side = "right"

    tr_ids, va_ids, te_ids = (encode(tokenizer, df, cfg) for df in (train_df, val_df, test_df))
    n_tok = np.array([len(x) for x in tr_ids])
    truncated = float(np.mean(n_tok >= cfg.max_len))
    print(f"[{cfg.name}] model={path} | mean tokens {n_tok.mean():.0f} | truncated {truncated:.1%}")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    model = load_llm(path, cfg, tokenizer) if is_llm else load_encoder(path)
    if cfg.freeze_embeddings and not is_llm:
        model.get_input_embeddings().weight.requires_grad_(False)
    single_gpu = is_llm or cfg.n_gpu == 1
    n_gpu = torch.cuda.device_count()
    devices = 1 if single_gpu else max(n_gpu, 1)
    steps_per_epoch = math.ceil(len(tr_ids) / (cfg.batch_size * cfg.grad_accum * devices))
    out_dir = os.path.join(WORK_DIR, f"tmp_{cfg.name}_{seed}")
    probe = np.random.default_rng(seed).choice(len(tr_ids), min(cfg.train_probe, len(tr_ids)), replace=False)
    args = make_training_args(
        output_dir=out_dir,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * (2 if is_llm else 4),
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs,
        warmup_steps=math.ceil(cfg.warmup_ratio * steps_per_epoch * cfg.epochs),
        weight_decay=cfg.weight_decay,
        lr_scheduler_type="cosine" if is_llm else "linear",
        fp16=cfg.fp16 and torch.cuda.is_available(),
        gradient_checkpointing=cfg.grad_ckpt and not is_llm,
        gradient_checkpointing_kwargs={"use_reentrant": False} if cfg.grad_ckpt and not is_llm else None,
        optim="paged_adamw_8bit" if is_llm else "adamw_torch",
        max_grad_norm=1.0,
        eval_strategy="steps",
        eval_steps=max(1, steps_per_epoch // cfg.evals_per_epoch),
        save_strategy="no",
        logging_steps=max(1, steps_per_epoch // 20),
        report_to="none",
        seed=seed,
        data_seed=seed,
        dataloader_num_workers=2,
        **LENGTH_GROUPING,
        label_names=["labels"],
    )
    if single_gpu and args.n_gpu > 1:
        args._n_gpu = 1
    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=to_dataset(tr_ids, y_train),
        # same loss on a fixed train sample and on the validation fold -> over/under-fitting is visible
        eval_dataset={
            "val": to_dataset(va_ids, y_val),
            "train": to_dataset([tr_ids[i] for i in probe], y_train[probe]),
        },
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=eval_metrics,
        callbacks=[HealthCallback()],
        class_weights=class_weights(y_train, cfg.class_weight),
        label_smoothing=cfg.label_smoothing,
    )
    trainer.train()
    infer_bs = cfg.batch_size * (2 if is_llm else 4)
    val_probs, test_probs = (
        predict_probs(trainer.model, tokenizer, x, infer_bs, cfg.fp16) for x in (va_ids, te_ids)
    )
    # guard: the final predictions must agree with the in-training evaluation of the same model
    evals = [r["eval_val_macro_f1"] for r in trainer.state.log_history if "eval_val_macro_f1" in r]
    final_f1 = float(f1_score(y_val, val_probs.argmax(1), average="macro", labels=list(range(len(LABELS)))))
    if evals and final_f1 < evals[-1] - 0.05:
        raise RuntimeError(
            f"prediction misalignment: in-training eval macro F1 {evals[-1]:.4f} vs final {final_f1:.4f}"
        )
    info = dict(
        model_path=path,
        minutes=(time.time() - t0) / 60,
        mean_tokens=float(n_tok.mean()),
        truncated=truncated,
        steps=trainer.state.global_step,
        devices=devices,
        peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0,
        history=trainer.state.log_history,
    )

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return val_probs, test_probs, info

In [ ]:
%%writefile src/rmitguard/postprocess.py
"""Decision rule tuned for macro F1.

argmax of the probabilities maximises accuracy, not macro F1. With PROMPT_INJECTION at 7% the model is
under-confident on it, so adding a per-class offset to the log-probabilities before argmax usually gains
a few points of macro F1 for free. Offsets are fitted on out-of-fold predictions only, never on the test set.
"""

from __future__ import annotations

import numpy as np

from .metrics import macro_f1

DEFAULT_GRID = np.round(np.arange(-3.0, 3.0001, 0.05), 4)


def predict_with_bias(probs: np.ndarray, bias: np.ndarray | None = None) -> np.ndarray:
    logp = np.log(np.clip(probs, 1e-9, 1.0))
    return np.argmax(logp if bias is None else logp + bias, axis=1)


def fit_bias(
    probs: np.ndarray, y: np.ndarray, grid: np.ndarray = DEFAULT_GRID, rounds: int = 3
) -> tuple[np.ndarray, float]:
    """Coordinate ascent on the offsets of classes 1..C-1 (class 0 is the reference)."""
    bias = np.zeros(probs.shape[1])
    best = macro_f1(y, predict_with_bias(probs, bias))
    for _ in range(rounds):
        improved = False
        for c in range(1, probs.shape[1]):
            for b in grid:
                trial = bias.copy()
                trial[c] = b
                score = macro_f1(y, predict_with_bias(probs, trial))
                if score > best + 1e-9:
                    best, bias, improved = score, trial, True
        if not improved:
            break
    return bias, best


def cross_fit_bias_score(probs: np.ndarray, y: np.ndarray, folds: np.ndarray) -> float:
    """Honest estimate of the tuned rule: fit offsets on the other folds, score on the held-out fold."""
    pred = np.empty(len(y), dtype=int)
    for k in np.unique(folds):
        va = folds == k
        bias, _ = fit_bias(probs[~va], y[~va])
        pred[va] = predict_with_bias(probs[va], bias)
    return macro_f1(y, pred)

In [ ]:
%%writefile src/rmitguard/runner.py
"""Run one experiment: K-fold CV -> OOF + averaged test probabilities -> report + submission.

Two protocols:
- full    (holdout: 0)   5-fold CV on all of train.csv; used for the models we submit.
- holdout (holdout: 0.2) 20% of train is locked away as a private-leaderboard stand-in; CV runs on the other
                         80% (dev). Decision offsets are fitted on dev OOF only, then the holdout is scored once.
                         Used to benchmark backbones on the same 2,800 rows.

Artifacts are rewritten after every fold, so a Kaggle session that hits its time limit still leaves
usable output, and teammates can each run a subset of folds and merge them with `rmitguard ensemble`.
"""

from __future__ import annotations

import json
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from .config import ExperimentConfig
from .constants import LABELS
from .data import CompetitionData, load_competition, make_split, split_fingerprint
from .metrics import bootstrap_ci, confusion_table, macro_f1, per_class_table, per_language_table
from .models import get_trainer
from .postprocess import cross_fit_bias_score, fit_bias, predict_with_bias
from .submission import label_distribution, make_submission, validate_submission


def subsample(data: CompetitionData, limit: int, seed: int) -> CompetitionData:
    """Stratified subset for smoke tests (`--limit 300`)."""
    train = data.train.groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(limit * len(g) // len(data.train), 10)), random_state=seed)
    )
    return CompetitionData(
        train=train.reset_index(drop=True), test=data.test.head(limit).reset_index(drop=True)
    )


def run_name(cfg: ExperimentConfig) -> str:
    return f"{cfg.name}-ho" if cfg.holdout else cfg.name


def score_oof(oof: np.ndarray, y: np.ndarray, folds: np.ndarray, language: np.ndarray) -> dict:
    """Scores on the rows that have OOF predictions (holdout rows never do)."""
    m = ~np.isnan(oof).any(axis=1)
    if not m.any():
        return {}
    probs, yy = oof[m], y[m]
    bias, tuned = fit_bias(probs, yy)
    pred = predict_with_bias(probs, bias)
    out = dict(
        coverage=float(m.sum() / (folds >= 0).sum()),
        macro_f1_argmax=macro_f1(yy, probs.argmax(1)),
        macro_f1_tuned=tuned,
        bias=bias.tolist(),
        public_lb_ci95=bootstrap_ci(yy, pred, n=500, sample_size=1800),
        per_class=per_class_table(yy, pred),
        per_language=per_language_table(yy, pred, language[m]),
        confusion=confusion_table(yy, pred),
    )
    if len(np.unique(folds[m])) >= 2:
        out["macro_f1_tuned_crossfit"] = cross_fit_bias_score(probs, yy, folds[m])
    return out


def score_holdout(probs: np.ndarray, y: np.ndarray, language: np.ndarray, bias: np.ndarray) -> dict:
    """The holdout is scored with offsets learnt on dev only — an honest private-LB estimate."""
    pred = predict_with_bias(probs, bias)
    return dict(
        n=len(y),
        macro_f1_argmax=macro_f1(y, probs.argmax(1)),
        macro_f1=macro_f1(y, pred),
        ci95=bootstrap_ci(y, pred, n=1000),
        public_lb_ci95=bootstrap_ci(y, pred, n=500, sample_size=1800),
        per_class=per_class_table(y, pred),
        per_language=per_language_table(y, pred, language),
        confusion=confusion_table(y, pred),
    )


def write_report(
    run_dir: Path,
    title: str,
    header: list[str],
    scores: dict,
    sub: pd.DataFrame | None,
    fold_stats: list[dict],
    holdout: dict | None = None,
) -> None:
    lines = [f"# {title}", "", *(f"- {h}" for h in header), ""]
    if holdout:
        lo, hi = holdout["ci95"]
        lines += [
            f"## Holdout ({holdout['n']} rows, never trained on, offsets from dev)",
            "| metric | value |",
            "|---|---|",
            f"| **holdout macro F1 ← benchmark score** | **{holdout['macro_f1']:.4f}** (95% CI {lo:.4f} – {hi:.4f}) |",
            f"| holdout macro F1 (argmax, no offsets) | {holdout['macro_f1_argmax']:.4f} |",
            "",
            holdout["per_class"].round(4).to_markdown(),
            "",
            holdout["per_language"].round(4).to_markdown(),
            "",
            holdout["confusion"].to_markdown(),
            "",
        ]
    if scores:
        lines += [
            "## Dev out-of-fold" if holdout else "## Out-of-fold (CV)",
            "| metric | value |",
            "|---|---|",
            f"| OOF coverage | {scores['coverage']:.1%} |",
            f"| macro F1 (argmax) | {scores['macro_f1_argmax']:.4f} |",
            f"| macro F1 (tuned offsets, in-sample) | {scores['macro_f1_tuned']:.4f} |",
        ]
        if "macro_f1_tuned_crossfit" in scores:
            lines.append(
                f"| **macro F1 (tuned, cross-fitted) ← trust this** | **{scores['macro_f1_tuned_crossfit']:.4f}** |"
            )
        lo, hi = scores["public_lb_ci95"]
        lines += [
            f"| public-LB-sized noise (95% CI, 1800 rows) | {lo:.4f} – {hi:.4f} |",
            f"| offsets (log-prob) | {np.round(scores['bias'], 2).tolist()} |",
            "",
            "### Per class",
            scores["per_class"].round(4).to_markdown(),
            "",
            "### Per language",
            scores["per_language"].round(4).to_markdown(),
            "",
            "### Confusion (OOF, tuned)",
            scores["confusion"].to_markdown(),
            "",
        ]
    if sub is not None:
        dist = label_distribution(sub)
        lines += ["## Test prediction distribution", *(f"- {k}: {v:.1%}" for k, v in dist.items()), ""]
    if fold_stats:
        lines += ["## Folds", pd.DataFrame(fold_stats).round(4).to_markdown(index=False)]
    (run_dir / "report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def _jsonable(scores: dict | None) -> dict | None:
    if scores is None:
        return None
    return {k: (v.round(5).to_dict() if isinstance(v, pd.DataFrame) else v) for k, v in scores.items()}


def predictions_frame(df: pd.DataFrame, probs: np.ndarray, pred: np.ndarray) -> pd.DataFrame:
    """Per-row predictions for error analysis (`charts.py`, notebooks)."""
    out = df[["id", "language", "label"]].copy()
    out["pred"] = np.asarray(LABELS)[pred]
    for i, lab in enumerate(LABELS):
        out[f"p_{lab}"] = probs[:, i].round(5)
    out["n_chars"] = df["text"].str.len().to_numpy()
    out["text"] = df["text"].str.slice(0, 400).to_numpy()
    return out


def run_experiment(
    cfg: ExperimentConfig,
    data_dir: str | None = None,
    out_dir: str | Path = "artifacts",
    limit: int | None = None,
) -> Path:
    t_start = time.time()
    data = load_competition(data_dir)
    if limit:
        data = subsample(data, limit, cfg.split_seed)
    train, test, y = data.train, data.test, data.y
    folds = make_split(train, cfg.n_folds, cfg.split_seed, cfg.holdout)
    split_fp = split_fingerprint(folds)
    language = train["language"].astype(str).to_numpy()
    hold = folds == -1
    hold_df = train[hold]
    # holdout rows are predicted together with the test set (no change to the model interface)
    predict_df = pd.concat([hold_df, test], ignore_index=True) if hold.any() else test
    n_hold = int(hold.sum())

    name = run_name(cfg)
    run_dir = Path(out_dir) / name
    run_dir.mkdir(parents=True, exist_ok=True)
    oof = np.full((len(train), len(LABELS)), np.nan, dtype=np.float32)
    pred_sum = np.zeros((len(predict_df), len(LABELS)), dtype=np.float64)
    fold_stats, history, done = [], [], []
    train_fold = get_trainer(cfg.kind)
    print(
        f"[{name}] {int((folds >= 0).sum())} dev / {n_hold} holdout / {len(test)} test | "
        f"folds {cfg.fold_ids()} | split {split_fp}"
    )

    for k in cfg.fold_ids():
        elapsed_h = (time.time() - t_start) / 3600
        if fold_stats and elapsed_h + fold_stats[-1]["minutes"] / 60 > cfg.time_budget_hours:
            print(f"⏭️ stopping before fold {k}: time budget {cfg.time_budget_hours}h would be exceeded")
            break
        tr, va = (folds != k) & (folds >= 0), folds == k
        val_p, pred_p, info = train_fold(
            cfg, train[tr], y[tr], train[va], y[va], predict_df, seed=cfg.seed + k
        )
        oof[va] = val_p
        pred_sum += pred_p
        done.append(k)
        history += [dict(fold=k, **row) for row in info.pop("history", [])]
        fold_stats.append(
            dict(
                fold=k,
                n_train=int(tr.sum()),
                n_val=int(va.sum()),
                macro_f1=macro_f1(y[va], val_p.argmax(1)),
                minutes=info.get("minutes", (time.time() - t_start) / 60),
                **{k2: v for k2, v in info.items() if k2 != "minutes"},
            )
        )
        print(f"[{name}] fold {k}: macro F1 (argmax) = {fold_stats[-1]['macro_f1']:.4f}")

        # persist after every fold
        pred_mean = (pred_sum / len(done)).astype(np.float32)
        hold_p, test_mean = pred_mean[:n_hold], pred_mean[n_hold:]
        scores = score_oof(oof, y, folds, language)
        bias = np.asarray(scores["bias"])
        holdout = score_holdout(hold_p, y[hold], language[hold], bias) if n_hold else None
        sub = make_submission(test["id"], predict_with_bias(test_mean, bias))
        problems = validate_submission(sub, test["id"], expected_rows=len(test))
        if problems:
            raise RuntimeError(f"invalid submission: {problems}")
        meta = dict(
            name=name,
            model=cfg.model,
            hf_id=cfg.hf_id,
            config=cfg.to_dict(),
            split_fp=split_fp,
            n_train=len(train),
            n_holdout=n_hold,
            n_test=len(test),
            folds_done=done,
            n_test_models=len(done),
            fold_stats=fold_stats,
            scores=_jsonable(scores),
            holdout=_jsonable(holdout),
            elapsed_minutes=round((time.time() - t_start) / 60, 2),
            updated_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        )
        np.save(run_dir / "oof.npy", oof)
        np.save(run_dir / "test.npy", test_mean)
        np.save(run_dir / "folds.npy", folds)
        if n_hold:
            np.save(run_dir / "holdout.npy", hold_p)
            predictions_frame(hold_df, hold_p, predict_with_bias(hold_p, bias)).to_csv(
                run_dir / "holdout_predictions.csv", index=False
            )
        m = ~np.isnan(oof).any(axis=1)
        predictions_frame(train[m], oof[m], predict_with_bias(oof[m], bias)).to_csv(
            run_dir / "oof_predictions.csv", index=False
        )
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        (run_dir / "meta.json").write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
        sub.to_csv(run_dir / "submission.csv", index=False)
        write_report(
            run_dir,
            name,
            [
                f"model: `{cfg.model}` ({cfg.hf_id or 'local diagnostic, not an approved backbone'}) · split `{split_fp}`",
                f"folds done: {done} / {cfg.n_folds} · {meta['elapsed_minutes']} min · updated {meta['updated_at']}",
            ],
            scores,
            sub,
            fold_stats,
            holdout,
        )

    try:  # charts must never cost us a finished run
        from .charts import plot_run

        plot_run(run_dir)
    except Exception:
        print(f"⚠️ charts failed:\n{traceback.format_exc()}")
    print((run_dir / "report.md").read_text(encoding="utf-8"))
    return run_dir

In [ ]:
%%writefile src/rmitguard/submission.py
"""Build and validate submission files. A malformed file burns one of only 6 daily submissions."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .constants import LABELS, N_TEST_ROWS


def make_submission(test_ids: pd.Series | np.ndarray, pred: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame({"id": np.asarray(test_ids), "label": np.asarray(LABELS)[pred]})


def validate_submission(
    sub: pd.DataFrame | str | Path,
    test_ids: pd.Series | np.ndarray | None = None,
    expected_rows: int | None = N_TEST_ROWS,
) -> list[str]:
    """Return a list of problems; empty list = safe to submit."""
    if not isinstance(sub, pd.DataFrame):
        sub = pd.read_csv(sub)
    problems = []
    if list(sub.columns) != ["id", "label"]:
        problems.append(f"columns must be exactly ['id', 'label'], got {list(sub.columns)}")
        return problems
    if expected_rows is not None and len(sub) != expected_rows:
        problems.append(f"expected {expected_rows} rows, got {len(sub)}")
    if sub["id"].duplicated().any():
        problems.append(f"{int(sub['id'].duplicated().sum())} duplicated ids")
    bad = sorted(set(sub["label"].astype(str)) - set(LABELS))
    if bad:
        problems.append(f"invalid labels {bad}")
    if test_ids is not None:
        missing = set(np.asarray(test_ids)) - set(sub["id"])
        extra = set(sub["id"]) - set(np.asarray(test_ids))
        if missing:
            problems.append(f"{len(missing)} test ids missing")
        if extra:
            problems.append(f"{len(extra)} ids not in test.csv")
    return problems


def label_distribution(sub: pd.DataFrame) -> dict[str, float]:
    return {lab: float((sub["label"] == lab).mean()) for lab in LABELS}

In [ ]:
%%writefile src/rmitguard/xling.py
"""Cross-lingual near-duplicates: the same source prompt appears translated into several languages.

Embeds every prompt with the *pre-trained* mmBERT-base (an approved backbone, no fine-tuning), removes each
language's mean vector, and finds for every row its most similar train row written in another language.
When that similarity is very high the two prompts are translations of each other and share the label
(99.5% of train pairs at similarity >= 0.80), so the neighbour's label overrides the model's prediction.

    python -m rmitguard xling --out artifacts/xling      # writes xling.npz (≈3 min on a T4)
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

from .constants import APPROVED_MODELS

LAYER = 11  # middle layer: more language-neutral than the last one
THRESHOLD = 0.78  # fold 0: fixes 3 errors, breaks none (0.75 breaks 1, 0.72 breaks 8)


def embed(
    texts: list[str], model_path: str, layer: int = LAYER, max_len: int = 256, batch: int = 64
) -> np.ndarray:
    import torch
    from transformers import AutoModel, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path, output_hidden_states=True).to(device).eval()
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(order), batch):
            idx = order[start : start + batch]
            enc = tok(
                [texts[i] for i in idx],
                truncation=True,
                max_length=max_len,
                padding=True,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            hidden = model(**enc).hidden_states[layer]
            mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            out[idx] = ((hidden * mask).sum(1) / mask.sum(1)).float().cpu().numpy()
    return out


def centre_by_language(emb: np.ndarray, language: np.ndarray) -> np.ndarray:
    emb = emb.copy()
    for lang in np.unique(language):
        m = language == lang
        emb[m] -= emb[m].mean(0)
    return emb / np.linalg.norm(emb, axis=1, keepdims=True)


def nearest_other_language(
    query: np.ndarray,
    q_lang: np.ndarray,
    ref: np.ndarray,
    r_lang: np.ndarray,
    allowed: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Index into `ref` and cosine similarity of the closest row in another language.

    `allowed` (n_query × n_ref bool, optional) removes candidates, e.g. rows of the same CV fold.
    """
    sim = query @ ref.T
    sim[q_lang[:, None] == r_lang[None, :]] = -1.0
    if allowed is not None:
        sim[~allowed] = -1.0
    return sim.argmax(1), sim.max(1)


def model_path(key: str) -> str:
    """Kaggle input mount if attached, otherwise the Hugging Face id (same rule as the trainer)."""
    import glob
    import os

    hf_id, handle, _ = APPROVED_MODELS[key]
    slug = handle.split("/")[1]
    hits = [
        os.path.dirname(p)
        for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
        if f"/{slug}/" in p.lower()
    ]
    return sorted(hits, key=len)[0] if hits else hf_id


def run_xling(data_dir: str | None, out_dir: str | Path, n_folds: int = 5, split_seed: int = 42) -> Path:
    from .data import load_competition, make_split

    data = load_competition(data_dir)
    train, test = data.train, data.test
    texts = train["text"].tolist() + test["text"].tolist()
    lang = np.concatenate([train["language"].to_numpy(dtype=str), test["language"].to_numpy(dtype=str)])
    path = model_path("mmbert-base")
    emb = centre_by_language(embed(texts, path), lang)
    e_tr, e_te = emb[: len(train)], emb[len(train) :]
    l_tr, l_te = lang[: len(train)], lang[len(train) :]
    folds = make_split(train, n_folds, split_seed)
    # train rows: neighbours only from OTHER folds -> usable on out-of-fold predictions without leakage
    tr_nn, tr_sim = nearest_other_language(e_tr, l_tr, e_tr, l_tr, allowed=folds[:, None] != folds[None, :])
    te_nn, te_sim = nearest_other_language(e_te, l_te, e_tr, l_tr)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    np.savez(
        out / "xling.npz",
        train_nn=tr_nn,
        train_sim=tr_sim,
        test_nn=te_nn,
        test_sim=te_sim,
        folds=folds,
        model=str(APPROVED_MODELS["mmbert-base"][0]),
        layer=LAYER,
    )
    print(
        f"xling: test rows with a translation (sim >= {THRESHOLD}): {(te_sim >= THRESHOLD).sum()} / {len(test)}"
    )
    return out / "xling.npz"


def apply_override(
    probs: np.ndarray, sim: np.ndarray, nn_label: np.ndarray, threshold: float = THRESHOLD
) -> np.ndarray:
    """Replace the probabilities of rows with a near-certain translation by the neighbour's one-hot label."""
    out = probs.copy()
    m = sim >= threshold
    out[m] = np.eye(probs.shape[1], dtype=probs.dtype)[nn_label[m]]
    return out

In [ ]:
import os, sys, importlib
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, 'src')
import rmitguard
importlib.reload(rmitguard)
import torch, transformers
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      '| torch', torch.__version__, '| transformers', transformers.__version__)

## Run every config on its own GPU

In [ ]:
JOBS = [('mmbert-small-f012', '# Small ModernBERT (384 hidden, 22 layers): fast multilingual reference.\nname: mmbert-small\nmodel: mmbert-small\nmax_len: 512\nhead_len: 384\nlr: 5.0e-5\nepochs: 3\nbatch_size: 32\ngrad_accum: 1\nclass_weight: sqrt\n', '0,1,2'), ('mmbert-small-f34', '# Small ModernBERT (384 hidden, 22 layers): fast multilingual reference.\nname: mmbert-small\nmodel: mmbert-small\nmax_len: 512\nhead_len: 384\nlr: 5.0e-5\nepochs: 3\nbatch_size: 32\ngrad_accum: 1\nclass_weight: sqrt\n', '3,4')]  # (job id, config yaml, folds)
HOLDOUT = None

import os, subprocess, sys, time
from pathlib import Path
import torch

Path('configs_run').mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)
FOLDS = {job: folds for job, _, folds in JOBS}
for job, text, _ in JOBS:
    if text:
        Path(f'configs_run/{job}.yaml').write_text(text)
pending = [job for job, _, _ in JOBS]
gpus = list(range(max(1, torch.cuda.device_count())))
running, status, t0 = {}, {}, time.time()


def launch(gpu, name):
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONPATH': 'src'}
    cmd = [sys.executable, '-m', 'rmitguard', 'train', f'configs_run/{name}.yaml', '--out', f'artifacts/{name}']
    if name == 'xling':  # cross-lingual neighbours (pre-trained mmBERT-base embeddings), a few minutes on a T4
        cmd = [sys.executable, '-m', 'rmitguard', 'xling', '--out', 'artifacts/xling']
    elif FOLDS[name]:
        cmd += ['--folds', FOLDS[name]]
    if HOLDOUT:
        cmd += ['--holdout', str(HOLDOUT)]
    return subprocess.Popen(cmd, env=env, stdout=open(f'logs/{name}.log', 'w'), stderr=subprocess.STDOUT)


while pending or running:
    for gpu in gpus:
        if gpu in running and running[gpu][1].poll() is not None:
            name, proc = running.pop(gpu)
            status[name] = proc.returncode
            print(f'[{(time.time() - t0) / 60:5.1f} min] {name} finished on GPU {gpu}: rc={proc.returncode}', flush=True)
        if gpu not in running and pending:
            name = pending.pop(0)
            running[gpu] = (name, launch(gpu, name))
            print(f'[{(time.time() - t0) / 60:5.1f} min] {name} started on GPU {gpu}', flush=True)
    time.sleep(20)
status

In [ ]:
import json, shutil
from pathlib import Path
for name, rc in status.items():
    print(f'===== {name} (rc={rc})')
    if rc != 0:
        print(''.join(Path(f'logs/{name}.log').read_text().splitlines(True)[-40:]))
        continue
    if name == 'xling':
        print(''.join(Path('logs/xling.log').read_text().splitlines(True)[-3:]))
        continue
    meta = json.loads(next(Path(f'artifacts/{name}').glob('*/meta.json')).read_text())
    s, ho = meta['scores'], meta.get('holdout')
    print('fold macro F1 (argmax):', [round(f['macro_f1'], 4) for f in meta['fold_stats']],
          '| tuned:', round(s['macro_f1_tuned'], 4), '| holdout:', ho and round(ho['macro_f1'], 4),
          '| minutes:', [round(f['minutes'], 1) for f in meta['fold_stats']])
ok = [n for n, rc in status.items() if rc == 0]
if ok:
    shutil.copy(next(Path(f'artifacts/{ok[0]}').glob('*/submission.csv')), 'submission.csv')